In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import os

path = r'C:\Users\Anirudh\Desktop\olist_project\\'
os.chdir(path)

orders      = pd.read_csv('olist_orders_dataset.csv')
items       = pd.read_csv('olist_order_items_dataset.csv')
reviews     = pd.read_csv('olist_order_reviews_dataset.csv')
payments    = pd.read_csv('olist_order_payments_dataset.csv')
products    = pd.read_csv('olist_products_dataset.csv')
sellers     = pd.read_csv('olist_sellers_dataset.csv')
customers   = pd.read_csv('olist_customers_dataset.csv')
translation = pd.read_csv('product_category_name_translation.csv')

print("All tables loaded ✅")

All tables loaded ✅


In [2]:
print(orders.dtypes)

order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object


In [3]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

print("Date columns converted ✅")
print(orders.dtypes)

Date columns converted ✅
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [4]:
print(orders[date_cols].head())
print("\nEarliest order date:", orders['order_purchase_timestamp'].min())
print("Latest order date:  ", orders['order_purchase_timestamp'].max())

  order_purchase_timestamp   order_approved_at order_delivered_carrier_date  \
0      2017-10-02 10:56:33 2017-10-02 11:07:15          2017-10-04 19:55:00   
1      2018-07-24 20:41:37 2018-07-26 03:24:27          2018-07-26 14:31:00   
2      2018-08-08 08:38:49 2018-08-08 08:55:23          2018-08-08 13:50:00   
3      2017-11-18 19:28:06 2017-11-18 19:45:59          2017-11-22 13:39:59   
4      2018-02-13 21:18:39 2018-02-13 22:20:29          2018-02-14 19:46:34   

  order_delivered_customer_date order_estimated_delivery_date  
0           2017-10-10 21:25:13                    2017-10-18  
1           2018-08-07 15:27:45                    2018-08-13  
2           2018-08-17 18:06:29                    2018-09-04  
3           2017-12-02 00:28:42                    2017-12-15  
4           2018-02-16 18:17:02                    2018-02-26  

Earliest order date: 2016-09-04 21:15:19
Latest order date:   2018-10-17 17:30:18


In [5]:
print("=== NULLS PER TABLE ===\n")
tables = {
    'orders': orders, 'items': items, 'reviews': reviews,
    'payments': payments, 'products': products,
    'sellers': sellers, 'customers': customers
}

for name, df in tables.items():
    nulls = df.isnull().sum().sum()
    print(f"{name:15} → {nulls} nulls")

=== NULLS PER TABLE ===

orders          → 4908 nulls
items           → 0 nulls
reviews         → 145903 nulls
payments        → 0 nulls
products        → 2448 nulls
sellers         → 0 nulls
customers       → 0 nulls


In [6]:
print(orders.isnull().sum())

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [7]:
# Orders with no delivery date = not yet delivered, we drop them for delivery analysis
# Keep only delivered orders
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()

# Drop rows where actual delivery date is missing
orders_delivered.dropna(subset=['order_delivered_customer_date'], inplace=True)

print(f"Original orders      : {len(orders)}")
print(f"Delivered orders     : {len(orders_delivered)}")
print(f"Rows removed         : {len(orders) - len(orders_delivered)}")
print(f"Nulls remaining      : {orders_delivered.isnull().sum().sum()}")

Original orders      : 99441
Delivered orders     : 96470
Rows removed         : 2971
Nulls remaining      : 15


In [10]:
print(orders_delivered.isnull().sum()[orders_delivered.isnull().sum() > 0])

order_approved_at               14
order_delivered_carrier_date     1
dtype: int64


In [11]:
# Check which columns have nulls
null_cols = orders_delivered.isnull().sum()
null_cols = null_cols[null_cols > 0]
print("Columns with nulls:\n", null_cols)

# These 15 rows have missing delivery timestamps
# They represent 0.015% of data — safe to drop without impacting analysis
orders_delivered.dropna(inplace=True)

print(f"\nRows after null removal : {len(orders_delivered)}")
print(f"Nulls remaining         : {orders_delivered.isnull().sum().sum()}")

Columns with nulls:
 order_approved_at               14
order_delivered_carrier_date     1
dtype: int64

Rows after null removal : 96455
Nulls remaining         : 0


In [12]:
orders_delivered['actual_delivery_days'] = (
    orders_delivered['order_delivered_customer_date'] -
    orders_delivered['order_purchase_timestamp']
).dt.days

print(orders_delivered['actual_delivery_days'].describe())

count    96455.000000
mean        12.093100
std          9.551209
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: actual_delivery_days, dtype: float64


In [13]:
orders_delivered['estimated_delivery_days'] = (
    orders_delivered['order_estimated_delivery_date'] -
    orders_delivered['order_purchase_timestamp']
).dt.days

print(orders_delivered['estimated_delivery_days'].describe())

count    96455.000000
mean        23.370898
std          8.757334
min          2.000000
25%         18.000000
50%         23.000000
75%         28.000000
max        155.000000
Name: estimated_delivery_days, dtype: float64


In [14]:
orders_delivered['delay_days'] = (
    orders_delivered['actual_delivery_days'] -
    orders_delivered['estimated_delivery_days']
)

print(orders_delivered['delay_days'].describe())

count    96455.000000
mean       -11.277798
std         10.191719
min       -146.000000
25%        -16.000000
50%        -12.000000
75%         -7.000000
max        189.000000
Name: delay_days, dtype: float64


In [15]:
orders_delivered['is_late'] = (orders_delivered['delay_days'] > 0).astype(int)

late_count = orders_delivered['is_late'].sum()
total = len(orders_delivered)
sla_breach_rate = round((late_count / total) * 100, 2)

print(f"Total delivered orders : {total}")
print(f"Late orders            : {late_count}")
print(f"SLA Breach Rate        : {sla_breach_rate}%")
print(f"SLA Compliance Rate    : {round(100 - sla_breach_rate, 2)}%")

Total delivered orders : 96455
Late orders            : 7306
SLA Breach Rate        : 7.57%
SLA Compliance Rate    : 92.43%


In [16]:
orders_delivered['purchase_year']  = orders_delivered['order_purchase_timestamp'].dt.year
orders_delivered['purchase_month'] = orders_delivered['order_purchase_timestamp'].dt.month
orders_delivered['purchase_day']   = orders_delivered['order_purchase_timestamp'].dt.day_name()

print(orders_delivered[['order_purchase_timestamp',
                          'purchase_year',
                          'purchase_month',
                          'purchase_day']].head())

  order_purchase_timestamp  purchase_year  purchase_month purchase_day
0      2017-10-02 10:56:33           2017              10       Monday
1      2018-07-24 20:41:37           2018               7      Tuesday
2      2018-08-08 08:38:49           2018               8    Wednesday
3      2017-11-18 19:28:06           2017              11     Saturday
4      2018-02-13 21:18:39           2018               2      Tuesday


In [18]:
# Building Master Data Frame
master = orders_delivered.merge(customers, on='customer_id', how='left')
print("After customers merge:", master.shape)

After customers merge: (96455, 19)


In [19]:
master = master.merge(items, on='order_id', how='left')
print("After items merge:", master.shape)

After items merge: (110173, 25)


In [20]:
master = master.merge(sellers, on='seller_id', how='left')
print("After sellers merge:", master.shape)

After sellers merge: (110173, 28)


In [21]:
master = master.merge(products, on='product_id', how='left')
print("After products merge:", master.shape)

After products merge: (110173, 36)


In [22]:
master = master.merge(translation, on='product_category_name', how='left')
print("After translation merge:", master.shape)

After translation merge: (110173, 37)


In [23]:
reviews_clean = reviews.groupby('order_id').first().reset_index()
master = master.merge(reviews_clean[['order_id', 'review_score']],
                      on='order_id', how='left')
print("After reviews merge:", master.shape)

After reviews merge: (110173, 38)


In [24]:
print("\n=== MASTER DATAFRAME ===")
print("Shape             :", master.shape)
print("Columns           :", master.columns.tolist())
print("Nulls remaining   :", master.isnull().sum().sum())
master.head(3)


=== MASTER DATAFRAME ===
Shape             : (110173, 38)
Columns           : ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'actual_delivery_days', 'estimated_delivery_days', 'delay_days', 'is_late', 'purchase_year', 'purchase_month', 'purchase_day', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'review_score']
Nulls remaining   : 8601


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,actual_delivery_days,estimated_delivery_days,...,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8,15,...,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13,19,...,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9,26,...,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto,5.0


In [25]:
master.to_csv('olist_master.csv', index=False)
print("Master file saved ✅ — olist_master.csv")

Master file saved ✅ — olist_master.csv


In [ ]:
# ===================== DAY 2 OBSERVATIONS =====================
# 1. Total delivered orders after cleaning     = 96,455
# 2. SLA Breach Rate                           = 7.57%
# 3. SLA Compliance Rate                       = 92.43%
# 4. Average actual delivery days              = 
# 5. Average delay days                        = 10
# 6. Final master dataframe shape (rows/cols)  = (110173, 38)
# ==============================================================

In [2]:
import pandas as pd
import os

path = r'C:\Users\Anirudh\Desktop\olist_project\\'
os.chdir(path)

master = pd.read_csv('olist_master.csv')

# Create delay column for late orders only (positive values only)
master['delay_days_late'] = master.apply(
    lambda x: x['delay_days'] if x['is_late'] == 1 else None, axis=1
)

master.to_csv('olist_master.csv', index=False)
print("Done ✅")
print("Null count in delay_days_late:", master['delay_days_late'].isnull().sum())
print("Min value:", master['delay_days_late'].min())

Done ✅
Null count in delay_days_late: 102028
Min value: 1.0


In [3]:
import pandas as pd
import os

path = r'C:\Users\Anirudh\Desktop\olist_project\\'
os.chdir(path)

master = pd.read_csv('olist_master.csv')
print(master.columns.tolist())
print("\ndelay_days_late sample:")
print(master['delay_days_late'].dropna().head())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'actual_delivery_days', 'estimated_delivery_days', 'delay_days', 'is_late', 'purchase_year', 'purchase_month', 'purchase_day', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'review_score', 'delay_days_late']

delay_days_late sample:
20    12.0
25     9.0
42     7.0
58     1.0
59     1.0
Name: delay_days_late, dtype: float64


In [1]:
import pandas as pd
import os

path = r'C:\Users\Anirudh\Desktop\olist_project\\'
os.chdir(path)

master = pd.read_csv('olist_master.csv')

# Create a breach rate column
master['breach_rate_pct'] = master['is_late'] * 100

master.to_csv('olist_master.csv', index=False)
print("Done ✅")
print("Average breach rate:", master['breach_rate_pct'].mean().round(2))

Done ✅
Average breach rate: 7.39


In [1]:
import pandas as pd
import os

path = r'C:\Users\Anirudh\Desktop\olist_project\\'
os.chdir(path)

master = pd.read_csv('olist_master.csv')

seller_scorecard = master.groupby('seller_id').agg(
    total_orders  = ('is_late', 'count'),
    late_orders   = ('is_late', 'sum'),
).reset_index()

seller_scorecard['breach_rate'] = round(
    (seller_scorecard['late_orders'] / 
     seller_scorecard['total_orders']) * 100, 2
)

# Filter min 10 orders
seller_scorecard = seller_scorecard[
    seller_scorecard['total_orders'] >= 10
]

print(seller_scorecard.sort_values(
    'breach_rate', ascending=False).head(10))

                             seller_id  total_orders  late_orders  breach_rate
2053  b1b3948701c5c72445495bd161b83a4c            14            9        64.29
1669  8d899e15a5925f097cca50faa49b15e3            10            6        60.00
451   2709af9587499e95e803a6498a5a56e9            46           23        50.00
2282  c42fd8e4d47dfb18ce5222f2dd7752f9            11            5        45.45
2929  fce62094ffe6a4009188ec44e681dfdd            11            5        45.45
2270  c37b2059d4f90d4feead554e5246565e            16            7        43.75
646   38e6dada03429a47197d5d584d793b41            12            5        41.67
30    02d35243ea2e497335cd0f076b45675d            16            6        37.50
1780  973f21788dfab357250f69a8dcb7ddee            19            7        36.84
2375  cb41bfbcbda0aea354a834ab222f9a59            11            4        36.36
